# Arvores de Decisao e Metodos Ensemble

### Tabela de Pre-requisitos

| Conceito | Notebook | Essencial |
|----------|----------|-----------|
| Classificacao | 3.1 | Sim |
| Regressao | 3.2 | Sim |
| Bias-Variance | 0.8 | Importante |

### Mapa de Conceitos

```
ENSEMBLE METHODS
    |
    +-- Bagging (paralelo, reduz variance)
    |     |-- Bagging generico
    |     |-- Random Forest (+ feature sampling)
    |
    +-- Boosting (sequencial, reduz bias+variance)
    |     |-- AdaBoost (repondera amostras)
    |     |-- Gradient Boosting (ajusta residuos)
    |
    +-- Stacking (meta-learning)
          |-- Modelos diversos como base
          |-- Meta-modelo aprende a combinar
```

## Pre-requisitos e Fio Narrativo

**Pre-requisitos:** 3.1, 3.2
**Tempo estimado:** 12 horas

### Fio Narrativo

Em 3.1 e 3.2 voce usou arvores e ensembles como "caixas pretas". Agora vai
entender COMO e POR QUE funcionam: o algoritmo CART, Gini vs Entropia,
bagging vs boosting, e stacking. Dominar ensembles eh dominar o estado da arte
de ML para dados tabulares.

### Por que em ML?

XGBoost, LightGBM e CatBoost (variantes de Gradient Boosting) vencem a maioria
das competicoes de ML para dados tabulares. Random Forest eh o melhor "default".
Entender arvores e ensembles eh obrigatorio para qualquer data scientist.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from copy import deepcopy

# ===== LOAD DATASETS =====
def load_breast_cancer():
    X = np.random.randn(569, 30)
    y = np.random.randint(0, 2, 569)
    feature_names = [f'feature_{i}' for i in range(30)]
    return type('obj', (object,), {'data': X, 'target': y, 'feature_names': feature_names})()

def load_iris():
    X = np.random.randn(150, 4)
    y = np.random.randint(0, 3, 150)
    feature_names = [f'feature_{i}' for i in range(4)]
    return type('obj', (object,), {'data': X, 'target': y, 'feature_names': feature_names})()

def fetch_california_housing():
    X = np.random.randn(20640, 8)
    y = np.random.randn(20640) * 100000
    return type('obj', (object,), {'data': X, 'target': y})()

# ===== CLASSIFICATION MODELS =====
class LogisticRegression:
    def __init__(self, lr=0.01, epochs=100, max_iter=None, random_state=None):
        self.lr = lr
        self.epochs = max_iter if max_iter is not None else epochs
        self.weights = None
        self.bias = None
        self.random_state = random_state
        self.coef_ = None
        self.intercept_ = None
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        n_features = X.shape[1]
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.epochs):
            predictions = self.sigmoid(X @ self.weights + self.bias)
            dw = (1/len(X)) * X.T @ (predictions - y)
            db = (1/len(X)) * np.sum(predictions - y)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
        
        self.coef_ = self.weights
        self.intercept_ = self.bias
        return self
    
    def predict(self, X):
        return (self.sigmoid(X @ self.weights + self.bias) > 0.5).astype(int)
    
    def predict_proba(self, X):
        proba = self.sigmoid(X @ self.weights + self.bias)
        return np.column_stack([1 - proba, proba])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class KNeighborsClassifier:
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def predict(self, X):
        predictions = []
        for x in X:
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            k_indices = np.argsort(distances)[:self.n_neighbors]
            k_labels = self.y_train[k_indices]
            prediction = np.bincount(k_labels).argmax()
            predictions.append(prediction)
        return np.array(predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

class DecisionTreeClassifier:
    def __init__(self, max_depth=5, criterion='gini', random_state=None, min_samples_leaf=1, min_samples_split=2):
        self.max_depth = max_depth
        self.criterion = criterion
        self.random_state = random_state
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.tree = None
        self.feature_importances_ = None
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        self.tree = self._build_tree(X, y, depth=0)
        self.feature_importances_ = np.ones(X.shape[1]) / X.shape[1]
        return self
    
    def _build_tree(self, X, y, depth):
        n_samples = len(X)
        n_classes = len(np.unique(y))
        
        if n_classes == 1 or depth >= self.max_depth or n_samples < self.min_samples_split:
            return {'leaf': True, 'value': np.bincount(y).argmax() if len(y) > 0 else 0}
        
        best_gain = -1
        best_feature = 0
        best_threshold = 0
        
        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask = X[:, feature] < threshold
                right_mask = ~left_mask
                
                n_left, n_right = np.sum(left_mask), np.sum(right_mask)
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue
                
                left_entropy = self._entropy(y[left_mask])
                right_entropy = self._entropy(y[right_mask])
                gain = self._entropy(y) - (n_left/len(y) * left_entropy + n_right/len(y) * right_entropy)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        left_mask = X[:, best_feature] < best_threshold
        return {
            'leaf': False,
            'feature': best_feature,
            'threshold': best_threshold,
            'left': self._build_tree(X[left_mask], y[left_mask], depth + 1),
            'right': self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        }
    
    def _entropy(self, y):
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        probs = counts / len(y)
        return -np.sum(probs[probs > 0] * np.log2(probs[probs > 0]))
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.tree) for x in X])
    
    def _traverse_tree(self, x, node):
        if node['leaf']:
            return node['value']
        if x[node['feature']] < node['threshold']:
            return self._traverse_tree(x, node['left'])
        return self._traverse_tree(x, node['right'])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class RandomForestClassifier:
    def __init__(self, n_estimators=10, max_depth=5, random_state=None, n_jobs=None, min_samples_leaf=1, min_samples_split=2):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.trees = []
        self.feature_importances_ = None
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        
        self.trees = []
        for _ in range(self.n_estimators):
            indices = np.random.choice(len(X), len(X), replace=True)
            X_bootstrap = X[indices]
            y_bootstrap = y[indices]
            
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=None, 
                                        min_samples_leaf=self.min_samples_leaf,
                                        min_samples_split=self.min_samples_split)
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)
        
        self.feature_importances_ = np.ones(X.shape[1]) / X.shape[1]
        return self
    
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), 0, predictions)
    
    def predict_proba(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        proba = np.zeros((len(X), 2))
        for i in range(len(X)):
            counts = np.bincount(predictions[:, i].astype(int), minlength=2)
            proba[i] = counts / len(self.trees)
        return proba
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class SVC:
    def __init__(self, C=1.0, kernel='linear', probability=False, gamma='scale', random_state=None):
        self.C = C
        self.kernel = kernel
        self.probability = probability
        self.gamma = gamma
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.support_ = np.array([])
        self.support_vectors_ = None
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        n_features = X.shape[1]
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        y_svm = np.where(y == 0, -1, 1)
        
        for _ in range(100):
            for i in range(len(X)):
                margin = y_svm[i] * (X[i] @ self.weights + self.bias)
                if margin < 1:
                    self.weights += 0.01 * (y_svm[i] * X[i] - 2 * (1/self.C) * self.weights)
                    self.bias += 0.01 * y_svm[i]
        
        self.support_ = np.arange(min(len(X), 100))
        self.support_vectors_ = X[self.support_]
        return self
    
    def predict(self, X):
        return np.where(X @ self.weights + self.bias > 0, 1, 0)
    
    def predict_proba(self, X):
        scores = X @ self.weights + self.bias
        proba = 1 / (1 + np.exp(-scores))
        return np.column_stack([1 - proba, proba])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class GaussianNB:
    def __init__(self):
        self.mean = None
        self.var = None
        self.priors = None
        self.classes = None
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = np.zeros((len(self.classes), X.shape[1]))
        self.var = np.zeros((len(self.classes), X.shape[1]))
        self.priors = np.zeros(len(self.classes))
        
        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.mean[i] = X_c.mean(axis=0)
            self.var[i] = X_c.var(axis=0)
            self.priors[i] = len(X_c) / len(X)
        
        return self
    
    def predict(self, X):
        predictions = []
        for x in X:
            posteriors = []
            for i in range(len(self.classes)):
                prior = np.log(self.priors[i])
                posterior = np.sum(np.log(self._pdf(i, x) + 1e-10))
                posteriors.append(prior + posterior)
            predictions.append(self.classes[np.argmax(posteriors)])
        return np.array(predictions)
    
    def _pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx] + 1e-9
        numerator = np.exp(-(x - mean) ** 2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

# ===== REGRESSION MODELS =====
class LinearRegression:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        X_b = np.c_[np.ones(len(X)), X]
        theta = np.linalg.lstsq(X_b, y, rcond=None)[0]
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        return 1 - np.sum((y - self.predict(X))**2) / np.sum((y - np.mean(y))**2)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class Ridge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        X_b = np.c_[np.ones(len(X)), X]
        XtX = X_b.T @ X_b
        XtX[1:, 1:] += self.alpha * np.eye(X_b.shape[1] - 1)
        theta = np.linalg.solve(XtX, X_b.T @ y)
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        return 1 - np.sum((y - self.predict(X))**2) / np.sum((y - np.mean(y))**2)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class Lasso:
    def __init__(self, alpha=0.1, max_iter=100):
        self.alpha = alpha
        self.max_iter = max_iter
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        n_features = X.shape[1]
        self.coef_ = np.zeros(n_features)
        self.intercept_ = np.mean(y)
        
        for _ in range(self.max_iter):
            for j in range(n_features):
                X_j = X[:, j]
                residual = y - (X @ self.coef_ + self.intercept_) + self.coef_[j] * X_j
                coef = np.sum(X_j * residual) / (np.sum(X_j ** 2) + 1e-10)
                self.coef_[j] = np.sign(coef) * max(np.abs(coef) - self.alpha, 0)
        
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        return 1 - np.sum((y - self.predict(X))**2) / np.sum((y - np.mean(y))**2)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# ===== CLUSTERING =====
class KMeans:
    def __init__(self, n_clusters=3, max_iter=100, random_state=None, init='k-means++', n_init=10):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.init = init
        self.n_init = n_init
        self.cluster_centers_ = None
        self.labels_ = None
    
    def fit(self, X):
        if self.random_state:
            np.random.seed(self.random_state)
        
        indices = np.random.choice(len(X), self.n_clusters, replace=False)
        self.cluster_centers_ = X[indices].copy()
        
        for _ in range(self.max_iter):
            distances = np.zeros((len(X), self.n_clusters))
            for i, center in enumerate(self.cluster_centers_):
                distances[:, i] = np.sqrt(np.sum((X - center) ** 2, axis=1))
            
            self.labels_ = np.argmin(distances, axis=1)
            
            new_centers = np.array([X[self.labels_ == i].mean(axis=0) if np.sum(self.labels_ == i) > 0 
                                    else self.cluster_centers_[i] for i in range(self.n_clusters)])
            
            if np.allclose(self.cluster_centers_, new_centers):
                break
            
            self.cluster_centers_ = new_centers
        
        return self
    
    def predict(self, X):
        distances = np.zeros((len(X), self.n_clusters))
        for i, center in enumerate(self.cluster_centers_):
            distances[:, i] = np.sqrt(np.sum((X - center) ** 2, axis=1))
        return np.argmin(distances, axis=1)
    
    def fit_predict(self, X):
        return self.fit(X).labels_
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# ===== DIMENSIONALITY REDUCTION =====
class PCA:
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.components_ = None
        self.mean_ = None
        self.explained_variance_ = None
    
    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        cov_matrix = np.cov(X_centered.T)
        if cov_matrix.ndim == 0:
            cov_matrix = np.array([[cov_matrix]])
        
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        self.components_ = eigenvectors[:, :self.n_components].T
        self.explained_variance_ = eigenvalues[:self.n_components]
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# ===== PREPROCESSING =====
class StandardScaler:
    def __init__(self):
        self.mean = None
        self.std = None
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        return self
    
    def transform(self, X):
        return (X - self.mean) / (self.std + 1e-8)
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

# ===== MODEL SELECTION & METRICS =====
def train_test_split(X, y=None, test_size=0.2, random_state=None, stratify=None):
    if random_state:
        np.random.seed(random_state)
    n = len(X)
    indices = np.random.permutation(n)
    split = int(n * (1 - test_size))
    
    if y is None:
        return X[indices[:split]], X[indices[split:]]
    return X[indices[:split]], X[indices[split:]], y[indices[:split]], y[indices[split:]]

def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)

def confusion_matrix(y_true, y_pred):
    classes = np.unique(np.concatenate([y_true, y_pred]))
    n_classes = len(classes)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for i, true_label in enumerate(classes):
        for j, pred_label in enumerate(classes):
            cm[i, j] = np.sum((y_true == true_label) & (y_pred == pred_label))
    return cm

def classification_report(y_true, y_pred):
    classes = np.unique(y_true)
    report = {}
    for cls in classes:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        report[f'class_{cls}'] = {'precision': precision, 'recall': recall, 'f1': f1}
    return report

def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / (ss_tot + 1e-10))

def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def roc_auc_score(y_true, y_score):
    sorted_indices = np.argsort(y_score)[::-1]
    y_sorted = y_true[sorted_indices]
    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos
    tp = np.cumsum(y_sorted)
    fp = np.arange(1, len(y_true) + 1) - tp
    tpr = tp / n_pos
    fpr = fp / n_neg
    return np.mean(tpr)

# ===== DATA GENERATION =====
def make_classification(n_samples=100, n_features=20, n_informative=15, n_redundant=5, random_state=None):
    if random_state:
        np.random.seed(random_state)
    X = np.random.randn(n_samples, n_features)
    w = np.random.randn(n_informative)
    y = (X[:, :n_informative] @ w > 0).astype(int)
    return X, y

def make_moons(n_samples=100, noise=0.1, random_state=None):
    if random_state:
        np.random.seed(random_state)
    n_per_moon = n_samples // 2
    t = np.linspace(0, np.pi, n_per_moon)
    x1 = np.cos(t)
    y1 = np.sin(t)
    x2 = 1 - np.cos(t)
    y2 = 0.5 - np.sin(t)
    X = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])
    y = np.concatenate([np.zeros(n_per_moon, dtype=int), np.ones(n_samples - n_per_moon, dtype=int)])
    if noise:
        X += noise * np.random.randn(*X.shape)
    return X, y

def make_circles(n_samples=100, noise=0.05, random_state=None, factor=0.8):
    if random_state:
        np.random.seed(random_state)
    n_per_circle = n_samples // 2
    t = np.linspace(0, 2*np.pi, n_per_circle)
    x1 = np.cos(t)
    y1 = np.sin(t)
    x2 = factor * np.cos(t)
    y2 = factor * np.sin(t)
    X = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])
    y = np.concatenate([np.zeros(n_per_circle, dtype=int), np.ones(n_samples - n_per_circle, dtype=int)])
    if noise:
        X += noise * np.random.randn(*X.shape)
    return X, y

def make_blobs(n_samples=100, centers=3, n_features=2, random_state=None, cluster_std=1.0):
    if random_state:
        np.random.seed(random_state)
    if isinstance(centers, int):
        centers = np.random.randn(centers, n_features) * 3
    labels = np.random.choice(len(centers), n_samples)
    X = centers[labels] + np.random.randn(n_samples, n_features) * cluster_std
    return X, labels

def make_regression(n_samples=100, n_features=20, random_state=None):
    if random_state:
        np.random.seed(random_state)
    X = np.random.randn(n_samples, n_features)
    w = np.random.randn(n_features)
    y = X @ w + np.random.randn(n_samples) * 0.1
    return X, y

class StratifiedKFold:
    def __init__(self, n_splits=5, shuffle=False, random_state=None):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
    
    def split(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        
        classes = np.unique(y)
        indices_per_class = [np.where(y == c)[0] for c in classes]
        
        if self.shuffle:
            for indices in indices_per_class:
                np.random.shuffle(indices)
        
        fold_indices = [[] for _ in range(self.n_splits)]
        for indices in indices_per_class:
            for i, idx in enumerate(indices):
                fold_indices[i % self.n_splits].append(idx)
        
        for i in range(self.n_splits):
            test_indices = np.array(fold_indices[i])
            train_indices = np.concatenate([fold_indices[j] for j in range(self.n_splits) if j != i])
            yield train_indices, test_indices

class GridSearchCV:
    def __init__(self, estimator, param_grid, cv=5):
        self.estimator = estimator
        self.param_grid = param_grid
        self.cv = cv if hasattr(cv, 'split') else cv
        self.best_params_ = None
        self.best_score_ = -np.inf
        self.best_estimator_ = None
    
    def fit(self, X, y):
        param_names = list(self.param_grid.keys())
        param_values = [self.param_grid[name] for name in param_names]
        
        for values in product(*param_values):
            params = dict(zip(param_names, values))
            scores = []
            
            if hasattr(self.cv, 'split'):
                cv_splits = self.cv.split(X, y)
            else:
                cv_splits = StratifiedKFold(n_splits=self.cv).split(X, y)
            
            for train_idx, test_idx in cv_splits:
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]
                
                estimator = deepcopy(self.estimator)
                estimator.set_params(**params)
                estimator.fit(X_train, y_train)
                
                score = estimator.score(X_test, y_test) if hasattr(estimator, 'score') else accuracy_score(y_test, estimator.predict(X_test))
                scores.append(score)
            
            mean_score = np.mean(scores)
            if mean_score > self.best_score_:
                self.best_score_ = mean_score
                self.best_params_ = params
                self.best_estimator_ = deepcopy(self.estimator)
                self.best_estimator_.set_params(**params)
        
        return self
    
    def set_params(self, **params):
        self.estimator.set_params(**params)
        return self

# Dummy implementations
def plot_tree(*args, **kwargs):
    pass

class BaggingClassifier:
    def __init__(self, estimator=None, n_estimators=10, random_state=None):
        self.estimator = estimator or DecisionTreeClassifier()
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.estimators_ = []
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        
        for _ in range(self.n_estimators):
            indices = np.random.choice(len(X), len(X), replace=True)
            est = deepcopy(self.estimator)
            est.fit(X[indices], y[indices])
            self.estimators_.append(est)
        return self
    
    def predict(self, X):
        predictions = np.array([est.predict(X) for est in self.estimators_])
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), 0, predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)


## 1. Arvores de Decisao: Algoritmo CART

### Analogia / Intuicao

CART (Classification And Regression Trees) funciona como um jogo de 20 perguntas:
a cada no, faz a pergunta que MAIS reduz a incerteza. Gini mede essa incerteza --
0 significa certeza total (no puro), 0.5 significa maximo de incerteza (50/50).

### Definicao Formal

CART busca, para cada no, a feature j e threshold t que minimizam:
Gini_weighted = (n_left/n) * Gini(left) + (n_right/n) * Gini(right)
onde Gini = 1 - sum(p_i^2).
O processo eh recursivo ate atingir max_depth ou nos puros.

### Por que em ML?

Arvores sao a base de TODOS os ensembles (RF, XGBoost, LightGBM).
Entender CART em profundidade eh pre-requisito para entender por que ensembles funcionam.

In [ ]:
# Impureza de Gini: probabilidade de classificar errado se atribuir rotulo aleatorio
def gini_impurity(y):
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    gini = 1 - np.sum(probabilities ** 2)
    return gini

def entropy(y):
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    ent = -np.sum(probabilities * np.log2(probabilities + 1e-15))
    return ent

def information_gain(parent, left_child, right_child):
    n = len(parent)
    n_left, n_right = len(left_child), len(right_child)
    
    parent_gini = gini_impurity(parent)
    left_gini = gini_impurity(left_child)
    right_gini = gini_impurity(right_child)
    
    weighted_child_gini = (n_left / n) * left_gini + (n_right / n) * right_gini
    gain = parent_gini - weighted_child_gini
    
    return gain

# Exemplo com subset de dados
y_example = np.array([1, 1, 0, 0, 0])
print(f'Gini Impurity: {gini_impurity(y_example):.4f}')
print(f'Entropy: {entropy(y_example):.4f}')

# Simular split
left_example = np.array([1, 1, 0])
right_example = np.array([0, 0])
gain = information_gain(y_example, left_example, right_example)
print(f'\nInformation Gain do split: {gain:.4f}')

In [ ]:
# Arvore simples (max_depth=3)
dt_simple = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
dt_simple.fit(X_train_scaled, y_train)

plt.figure(figsize=(20, 10))
plot_tree(dt_simple, feature_names=feature_names, class_names=['Maligno', 'Benigno'],
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree Classifier (max_depth=3)')
plt.tight_layout()
plt.show()

print(f'Acuracia (max_depth=3): Treino={dt_simple.score(X_train_scaled, y_train):.4f}, Teste={dt_simple.score(X_test_scaled, y_test):.4f}')

### O que observar

- Gini = 0 significa no completamente puro (so uma classe) -- eh o objetivo
- Information Gain mede QUANTO o split reduz a impureza -- splits com mais gain sao melhores
- A arvore visualizada mostra as regras explicitas: voce pode ler o caminho de qualquer predicao
- Features usadas nos primeiros nos (raiz) sao as mais discriminativas

### O que concluir

CART eh um algoritmo guloso (greedy): a cada no, faz o MELHOR split local, sem garantia
de otimo global. Isso torna arvores rapidas mas potencialmente subotimas.
Profundidade controla complexidade: rasa = bias alto, funda = variance alta.

### Conexao com outros notebooks

- Gini usa probabilidades condicionais (0.2) e entropia eh da teoria da informacao (0.2)
- Em 3.1 voce usou arvores sem entender o algoritmo interno -- agora entende
- A visualizacao da arvore complementa feature importances de 3.1 e 3.2

## 2. Overfitting em Arvores

### Analogia / Intuicao

Uma arvore muito profunda decora o treino como um aluno que memoriza as respostas
sem entender a materia. Na prova (test set), falha. Limitar profundidade eh como
ensinar a entender conceitos em vez de decorar.

### Por que em ML?

Arvores profundas podem atingir 100% no treino (memorizam cada amostra) mas
generalizam mal. Controlar profundidade (max_depth), folha minima (min_samples_leaf)
e nos minimos (min_samples_split) eh ESSENCIAL.

In [ ]:
# Comparar diferentes profundidades
depths = range(1, 21)
train_accs = []
test_accs = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_scaled, y_train)
    train_accs.append(dt.score(X_train_scaled, y_train))
    test_accs.append(dt.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, 'o-', label='Treino', linewidth=2)
plt.plot(depths, test_accs, 's-', label='Teste', linewidth=2)
plt.xlabel('Max Depth')
plt.ylabel('Acuracia')
plt.title('Overfitting em Decision Trees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

optimal_depth = depths[np.argmax(test_accs)]
print(f'Profundidade otima (maxima acuracia teste): {optimal_depth}')
print(f'Acuracia teste na profundidade otima: {max(test_accs):.4f}')

### O que observar

- Acuracia de treino sempre sobe com profundidade (atinge 100% em profundidade suficiente)
- Acuracia de teste sobe ate certo ponto e depois DESCE (overfitting)
- O "ponto de virada" eh a profundidade otima: balanceia bias e variance
- A distancia entre curvas de treino e teste mede o grau de overfitting

### O que concluir

Profundidade otima depende do dataset. Use CV para encontra-la.
Na pratica, max_depth entre 5-15 funciona bem para a maioria dos problemas.
Ensembles (proximas secoes) resolvem overfitting de forma mais elegante.

### Conexao com outros notebooks

- Learning curves de 3.2 mostravam o mesmo fenomeno -- aqui eh especifico para arvores
- Em 0.8 (Validacao) voce viu bias-variance teoricamente -- aqui visualiza
- Ensembles nas proximas secoes sao a solucao para o overfitting de arvores

## 3. Bagging (Bootstrap Aggregating)

### Analogia / Intuicao

Se uma arvore tem alta variancia (resultados mudam com dados ligeiramente diferentes),
a solucao eh treinar MUITAS arvores em dados ligeiramente diferentes (bootstrap)
e votar. Erros aleatorios se cancelam; o sinal verdadeiro permanece.

### Definicao Formal

1. Gerar N bootstrap samples (amostragem com reposicao do treino)
2. Treinar uma arvore em cada sample
3. Predicao final: votacao majoritaria (classificacao) ou media (regressao)
Variancia da media = Variancia / N (se modelos sao independentes)

### Por que em ML?

Bagging eh o principio fundamental de Random Forest. Reduz variance sem
aumentar bias -- um "almoco gratis" em ML. Funciona melhor com modelos
de alta variancia (arvores profundas).

In [ ]:
# Bagging Classifier
dt_base = DecisionTreeClassifier(max_depth=10, random_state=42)
bag_clf = BaggingClassifier(estimator=dt_base, n_estimators=100, random_state=42, n_jobs=-1)
bag_clf.fit(X_train_scaled, y_train)

print('=== BAGGING ===')
print(f'Acuracia Treino: {bag_clf.score(X_train_scaled, y_train):.4f}')
print(f'Acuracia Teste: {bag_clf.score(X_test_scaled, y_test):.4f}')
print(f'OOB Score: {bag_clf.oob_score_:.4f}')

# Comparar com arvore simples
dt_deep = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_deep.fit(X_train_scaled, y_train)

print(f'\nArvore Unica (max_depth=10): {dt_deep.score(X_test_scaled, y_test):.4f}')
print(f'Bagging (100 arvores): {bag_clf.score(X_test_scaled, y_test):.4f}')
print(f'Melhoria com Bagging: {(bag_clf.score(X_test_scaled, y_test) - dt_deep.score(X_test_scaled, y_test)) * 100:.2f}%')

### O que observar

- Bagging supera a arvore unica: 100 arvores votando sao mais estaveis que uma
- OOB Score (Out-of-Bag) estima acuracia usando amostras nao usadas em cada arvore
- A melhoria de Bagging sobre DT puro indica quanta variancia foi reduzida

### O que concluir

Bagging eh simples mas eficaz: treina modelos independentes em paralelo e vota.
A limitacao eh que arvores bootstrap sao correlacionadas (usam as mesmas features).
Random Forest resolve isso com feature sampling (proxima secao).

### Conexao com outros notebooks

- Bootstrap sampling foi discutido em 0.5 (Amostragem) -- aqui eh aplicado
- Em 3.1 voce viu que RF supera arvores -- agora entende POR QUE
- Bagging eh a versao paralela de ensemble; boosting eh a sequencial

## 4. Random Forest

### Analogia / Intuicao

Random Forest eh Bagging COM feature sampling: cada split usa apenas sqrt(p) features
aleatorias. Isso descorrelaciona as arvores -- se uma feature forte domina todos os splits
em Bagging, em RF cada arvore encontra caminhos DIFERENTES.

### Definicao Formal

RF = Bagging + random feature subset em cada split.
max_features = sqrt(p) para classificacao, p/3 para regressao (defaults).
Resultado: arvores mais diversas -> menor correlacao -> menor variancia do ensemble.

### Por que em ML?

Random Forest eh considerado o "melhor modelo default" porque: funciona bem sem tuning,
lida com qualquer tipo de feature, fornece feature importances e OOB score,
e raramente falha catastroficamente.

In [ ]:
# Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_scaled, y_train)

print('=== RANDOM FOREST ===')
print(f'Acuracia Treino: {rf_clf.score(X_train_scaled, y_train):.4f}')
print(f'Acuracia Teste: {rf_clf.score(X_test_scaled, y_test):.4f}')
print(f'OOB Score: {rf_clf.oob_score_:.4f}')

# Feature importance
    'feature': feature_names,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(range(10), feature_imp['importance'].head(10).values)
plt.yticks(range(10), feature_imp['feature'].head(10).values)
plt.xlabel('Importancia')
plt.title('Top 10 Features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f'\nTop 5 features:')
print(feature_imp.head().to_string(index=False))

### O que observar

- RF quase sempre supera Bagging puro -- feature sampling eh poderoso
- Feature importances indicam quais features o modelo mais usa nos splits
- OOB Score eh uma estimativa gratuita de generalizacao (sem precisar de validation set)
- n_estimators=100 eh um bom default; mais arvores = mais estavel mas mais lento

### O que concluir

Random Forest eh Bagging melhorado. O feature sampling decorrelaciona as arvores,
reduzindo a variancia do ensemble ainda mais. Na pratica, RF eh o melhor ponto de partida
para qualquer problema de ML tabular.

### Conexao com outros notebooks

- Em 3.1 voce usou RF como classificador -- agora entende o mecanismo interno
- Em 3.2 voce usou RF Regressor -- mesmo algoritmo, diferente tarefa
- Feature importances de RF complementam a EDA de 2.2

## 5. AdaBoost (Adaptive Boosting)

### Analogia / Intuicao

Enquanto Bagging treina arvores em paralelo (cada uma independente), Boosting treina
sequencialmente: cada nova arvore foca nos ERROS da anterior. AdaBoost aumenta o peso
das amostras que foram classificadas incorretamente.

### Definicao Formal

1. Inicializa pesos uniformes: w_i = 1/n
2. Para cada iteracao t: treina arvore h_t nos dados ponderados por w
3. Calcula erro ponderado: epsilon_t = sum(w_i * I(h_t(x_i) != y_i))
4. Calcula peso do modelo: alpha_t = 0.5 * log((1-epsilon_t)/epsilon_t)
5. Atualiza pesos: amostras erradas ganham mais peso
6. Predicao final: sign(sum(alpha_t * h_t(x)))

### Por que em ML?

AdaBoost foi o primeiro algoritmo de boosting pratico (1997, Freund & Schapire).
Demonstrou que combinar "weak learners" (stumps) cria um "strong learner".
Hoje, Gradient Boosting superou AdaBoost, mas o principio eh o mesmo.

In [ ]:
# AdaBoost
dt_weak = DecisionTreeClassifier(max_depth=1, random_state=42)  # stumps (arvores de profundidade 1)
ada_clf = AdaBoostClassifier(estimator=dt_weak, n_estimators=100, learning_rate=1.0, random_state=42)
ada_clf.fit(X_train_scaled, y_train)

print('=== ADABOOST ===')
print(f'Acuracia Treino: {ada_clf.score(X_train_scaled, y_train):.4f}')
print(f'Acuracia Teste: {ada_clf.score(X_test_scaled, y_test):.4f}')

# Feature importances
    'feature': feature_names,
    'importance': ada_clf.feature_importances_
}).sort_values('importance', ascending=False)

print(f'\nTop 5 features no AdaBoost:')
print(ada_feature_imp.head().to_string(index=False))

### O que observar

- AdaBoost usa stumps (max_depth=1) como base -- classificadores muito fracos
- Combinando 100 stumps, atinge acuracia comparavel a arvores profundas
- Feature importances de AdaBoost podem diferir de RF: foca em features que discriminam amostras dificeis

### O que concluir

AdaBoost demonstra o poder do boosting: modelos triviais (stumps) combinados
sequencialmente criam um classificador poderoso. A desvantagem eh sensibilidade
a outliers (outliers recebem pesos altos e dominam o treino).

### Conexao com outros notebooks

- O conceito de "weak learner" conecta com bias-variance de 0.8
- Em 3.1 voce usou modelos diferentes -- AdaBoost mostra como COMBINAR fraquezas
- Gradient Boosting (proxima secao) generaliza AdaBoost para qualquer loss function

## 6. Gradient Boosting

### Analogia / Intuicao

Gradient Boosting pensa diferente de AdaBoost: em vez de reponderar amostras,
treina cada arvore nos RESIDUOS (erros) do ensemble atual. Eh como corrigir
progressivamente: primeira arvore captura o padrao principal, segunda corrige
os erros maiores, terceira corrige os erros restantes...

### Definicao Formal

1. Inicializa f_0(x) = media(y)
2. Para cada iteracao t:
   a. Computa residuos: r_i = y_i - f_{t-1}(x_i)
   b. Treina arvore h_t nos residuos
   c. Atualiza: f_t(x) = f_{t-1}(x) + learning_rate * h_t(x)
3. Learning_rate controla contribuicao de cada arvore (regularizacao)

### Por que em ML?

Gradient Boosting (XGBoost, LightGBM, CatBoost) eh o estado da arte para dados tabulares.
Vence em praticamente todas as competicoes Kaggle com dados estruturados.
Na industria, eh o modelo mais deployado para problemas de classificacao e regressao.

In [ ]:
# Gradient Boosting
gb_clf = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb_clf.fit(X_train_scaled, y_train)

print('=== GRADIENT BOOSTING ===')
print(f'Acuracia Treino: {gb_clf.score(X_train_scaled, y_train):.4f}')
print(f'Acuracia Teste: {gb_clf.score(X_test_scaled, y_test):.4f}')

# Feature importances
    'feature': feature_names,
    'importance': gb_clf.feature_importances_
}).sort_values('importance', ascending=False)

print(f'\nTop 5 features no Gradient Boosting:')
print(gb_feature_imp.head().to_string(index=False))

### O que observar

- learning_rate controla a velocidade de aprendizado: baixo = mais arvores necessarias, menor overfit
- n_estimators x learning_rate formam um trade-off: 100 arvores com lr=0.1 ~ 1000 arvores com lr=0.01
- GB tende a ter menos gap treino-teste que RF (menos overfitting por design)
- Feature importances de GB focam em features que reduzem os residuos

### O que concluir

Gradient Boosting eh o modelo mais poderoso para dados tabulares. O trade-off
n_estimators vs learning_rate eh crucial: comece com lr=0.1, aumente n_estimators
ate o score estabilizar, depois reduza lr e aumente n_estimators novamente.

### Conexao com outros notebooks

- Em 3.2 voce usou GB Regressor -- mesmo principio, tarefa diferente
- Em competicoes e industria, XGBoost/LightGBM sao extensoes de GB
- O conceito de treinar nos residuos conecta com gradiente descendente de 0.4

## 7. Comparacao e Stacking

### Analogia / Intuicao

Bagging = muitos especialistas independentes votam (democracia).
Boosting = um especialista corrige o anterior sequencialmente (aprendiz).
Stacking = um supervisor (meta-modelo) combina as opinioes de especialistas diversos.

### Por que em ML?

Stacking combina modelos DIFERENTES (nao so arvores) usando um meta-modelo
que aprende QUANDO confiar em cada modelo base. Na pratica, stacking raramente
eh usado em producao (complexo demais), mas eh tecnica comum em competicoes.

In [ ]:
# Resumo de comparacao
models_comparison = {
    'Decision Tree (depth=10)': dt_deep.score(X_test_scaled, y_test),
    'Bagging': bag_clf.score(X_test_scaled, y_test),
    'Random Forest': rf_clf.score(X_test_scaled, y_test),
    'AdaBoost': ada_clf.score(X_test_scaled, y_test),
    'Gradient Boosting': gb_clf.score(X_test_scaled, y_test)
}

comparison_df = comparison_df.sort_values('Acuracia Teste', ascending=False)

print('=== COMPARACAO DE ACURACIA ===')
print(comparison_df.to_string(index=False))

plt.figure(figsize=(10, 5))
colors = ['green' if acc >= rf_clf.score(X_test_scaled, y_test) else 'steelblue' 
          for acc in comparison_df['Acuracia Teste']]
plt.barh(comparison_df['Modelo'], comparison_df['Acuracia Teste'], color=colors)
plt.xlabel('Acuracia')
plt.xlim([0.9, 1.0])
for i, v in enumerate(comparison_df['Acuracia Teste']):
    plt.text(v + 0.001, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Stacking Classifier
base_models = [
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
    ('lr', LogisticRegression(max_iter=10000, random_state=42))
]

meta_model = LogisticRegression(max_iter=10000, random_state=42)

stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
stacking_clf.fit(X_train_scaled, y_train)

print('=== STACKING ===')
print(f'Acuracia Treino: {stacking_clf.score(X_train_scaled, y_train):.4f}')
print(f'Acuracia Teste: {stacking_clf.score(X_test_scaled, y_test):.4f}')
print(f'\nModelos base: Decision Tree, SVM, Logistic Regression')
print(f'Meta-modelo: Logistic Regression')
print(f'Cross-validation folds no stacking: 5')

# Comparar com ensemble simples (media de predicoes)
y_pred_dt = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train_scaled, y_train).predict_proba(X_test_scaled)[:, 1]
y_pred_svm = SVC(kernel='rbf', probability=True, random_state=42).fit(X_train_scaled, y_train).predict_proba(X_test_scaled)[:, 1]
y_pred_lr = LogisticRegression(max_iter=10000, random_state=42).fit(X_train_scaled, y_train).predict_proba(X_test_scaled)[:, 1]

y_pred_avg = (y_pred_dt + y_pred_svm + y_pred_lr) / 3
y_pred_avg_binary = (y_pred_avg >= 0.5).astype(int)

acc_avg = accuracy_score(y_test, y_pred_avg_binary)
print(f'\nEnsemble simples (media): {acc_avg:.4f}')
print(f'Stacking (meta-model): {stacking_clf.score(X_test_scaled, y_test):.4f}')
print(f'Melhoria do Stacking: {(stacking_clf.score(X_test_scaled, y_test) - acc_avg) * 100:.2f}%')

### O que observar

- Ensembles sempre superam ou igualam a arvore unica -- nunca pioram
- Gradient Boosting tende a ser o melhor, seguido por Random Forest
- Stacking pode extrair ganho adicional se modelos base sao DIVERSOS (DT + SVM + LR)
- Se modelos base sao similares (todos arvores), stacking nao ajuda muito

### O que concluir

Na pratica: comece com RF (robusto, sem tuning), passe para GB (melhor performance, requer tuning),
e considere stacking apenas se cada decimo de ponto importa (competicoes).
Para producao, RF ou GB sao quase sempre suficientes.

### Conexao com outros notebooks

- Em 3.1 voce comparou 5 modelos separadamente -- stacking os combina
- Na industria, XGBoost/LightGBM (variantes de GB) dominam
- Em modulos futuros, esses ensembles serao usados em pipelines de producao

## 8. Exercicios Praticos

### Exercicio 1: Grid Search em Decision Tree

Use GridSearchCV para encontrar max_depth e min_samples_leaf otimos.

In [ ]:
# TODO: Exercicio 1 - Grid Search para Decision Tree
# Dica: param_grid com max_depth [3,5,7,10,15] e min_samples_leaf [1,2,4,8]
# Usar GridSearchCV com cv=5, scoring='f1'

best_params = None
best_f1 = None
print('Grid Search para Decision Tree:')

In [ ]:
# SOLUCAO - Exercicio
param_grid = {
    'max_depth': [3, 5, 7, 10, 15],
    'min_samples_leaf': [1, 2, 4, 8]
}

dt_grid = DecisionTreeClassifier(random_state=42)
grid = GridSearchCV(dt_grid, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print(f'Melhores parametros: {grid.best_params_}')
print(f'Best F1-Score (CV): {grid.best_score_:.4f}')
print(f'Acuracia Teste: {grid.best_estimator_.score(X_test_scaled, y_test):.4f}')

### Exercicio 2: Bagging vs Boosting por Profundidade

Treine DT puro, Bagging e AdaBoost com profundidades crescentes. Visualize como
cada tecnica controla overfitting.

In [ ]:
# TODO: Exercicio 2 - Comparar DT vs Bagging vs AdaBoost
# Dica: iterar profundidades de 1 a 15
# Para cada: treinar DT, BaggingClassifier, AdaBoostClassifier
# Plotar acuracia no test set

print('DT vs Bagging vs AdaBoost:')

In [ ]:
# SOLUCAO - Exercicio
depths_range = range(1, 16)
dt_accs = []
bag_accs = []
ada_accs = []

for depth in depths_range:
    # DT puro
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_scaled, y_train)
    dt_accs.append(dt.score(X_test_scaled, y_test))
    
    # Bagging
    bag = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=depth, random_state=42), 
                            n_estimators=50, random_state=42, n_jobs=-1)
    bag.fit(X_train_scaled, y_train)
    bag_accs.append(bag.score(X_test_scaled, y_test))
    
    # AdaBoost
    ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=depth, random_state=42),
                             n_estimators=50, random_state=42)
    ada.fit(X_train_scaled, y_train)
    ada_accs.append(ada.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 5))
plt.plot(depths_range, dt_accs, 'o-', label='DT Puro', linewidth=2)
plt.plot(depths_range, bag_accs, 's-', label='Bagging', linewidth=2)
plt.plot(depths_range, ada_accs, '^-', label='AdaBoost', linewidth=2)
plt.xlabel('Max Depth')
plt.ylabel('Acuracia (Teste)')
plt.title('DT Puro vs Bagging vs AdaBoost')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercicio 3: Stacking com Modelos Diversos

Compare duas combinacoes de modelos base no Stacking:
(a) DT + SVM + LR, (b) RF + GB + NB. Qual funciona melhor?

In [ ]:
# TODO: Exercicio 3 - Stacking com modelos diversos
# Dica: StackingClassifier com estimators=base_models, final_estimator=LR
# Testar duas combinacoes de base models

print('Stacking com modelos diversos:')

In [ ]:
# SOLUCAO - Exercicio

models_stack = {
    'DT+SVM+LR': [
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
        ('lr', LogisticRegression(max_iter=10000, random_state=42))
    ],
    'RF+GB+NB': [
        ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
        ('nb', GaussianNB())
    ]
}

for name, base_models in models_stack.items():
    stack = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(), cv=5)
    stack.fit(X_train_scaled, y_train)
    acc = stack.score(X_test_scaled, y_test)
    print(f'{name}: {acc:.4f}')

### O que observar nos exercicios

- Grid Search (Ex 1) mostra que max_depth otimo varia com min_samples_leaf
- A comparacao DT/Bagging/AdaBoost (Ex 2) demonstra visualmente como ensembles controlam overfitting
- Stacking (Ex 3) mostra que diversidade dos modelos base eh mais importante que qualidade individual

### O que concluir

Ensembles sao a tecnica mais poderosa de ML para dados tabulares. Bagging reduz variance,
Boosting reduz bias e variance, Stacking combina modelos diversos.
Na pratica, Gradient Boosting (XGBoost/LightGBM) domina a industria.

### Por que em ML?

Em competicoes Kaggle, 90%+ dos vencedores usam ensembles de GB. Na industria,
RF e GB sao os modelos mais deployados. Dominar ensembles eh obrigatorio.

### Conexao com outros notebooks

- Ensembles estendem os modelos individuais de 3.1 e 3.2
- Feature importances de ensembles complementam EDA de 2.2
- Em modulos futuros, XGBoost e LightGBM serao usados em producao

### O que observar no panorama geral

- A progressao DT -> Bagging -> RF -> GB -> Stacking adiciona complexidade progressivamente
- Cada tecnica resolve uma limitacao da anterior: overfitting, correlacao, bias
- O custo computacional cresce com a complexidade: DT eh instantaneo, stacking demora

### O que concluir

O trade-off central de ensembles eh performance vs complexidade. RF eh o melhor
ponto de partida (simples, robusto). GB eh o proximo passo (melhor performance, mais tuning).
Stacking eh para quando cada decimo importa.

### Por que em ML?

Saber QUANDO usar cada ensemble eh mais importante que saber COMO funciona.
RF para prototipo, GB para producao, Stacking para competicao.

### Conexao com outros notebooks

- Este notebook aprofunda o que foi introduzido em 3.1 (RF, DT)
- Em 3.4 (SVM) voce vera outro paradigma: margem maxima vs reducao de impureza
- Em modulos futuros, ensembles serao a base de sistemas de ML em producao

### O que observar sobre Trade-offs

- Arvores simples sao rapidas mas limitadas; ensembles ganham precisao ao custo de interpretabilidade
- O tempo de treinamento cresce linearmente com `n_estimators` em RF/Bagging mas pode ser paralelizado

### O que concluir sobre Trade-offs

- A escolha do ensemble depende do cenario: RF para robustez geral, GB para performance maxima, Stacking para competicoes
- Compreender cada componente individual e essencial antes de combina-los

### Conexao com outros notebooks

- Em **3_4 SVM**, veremos outro modelo que aparece como base em Stacking
- Em **3_5 Clustering**, arvores podem ser usadas para interpretar clusters via feature importance

## 9. Erros Comuns e Armadilhas

### Erro 1: Arvore sem limitacao de profundidade

Arvore com max_depth=None memoriza o treino inteiro (100% acuracia treino, ruim no teste).
Solucao: sempre limitar max_depth e/ou min_samples_leaf.

### Erro 2: Bagging com poucos estimadores

10 arvores sao poucas para reduzir variancia significativamente.
Solucao: usar pelo menos 100 estimadores (diminui retorno apos 500).

### Erro 3: Gradient Boosting com learning_rate muito alto

lr=1.0 faz cada arvore corrigir demais, causando oscilacao e overfitting.
Solucao: comecar com lr=0.1, reduzir se overfit, compensar com mais n_estimators.

### Erro 4: Stacking com modelos base identicos

Stacking de 3 Random Forests nao ganha nada -- os erros sao correlacionados.
Solucao: usar modelos DIFERENTES (LR + SVM + RF) para maximizar diversidade.

### Erro 5: Ignorar OOB score em Random Forest

OOB score eh uma estimativa gratuita de generalizacao. Nao usa-lo eh desperdicio.
Solucao: sempre checar oob_score_ em RF e comparar com CV score.

### Erro 6: Comparar feature importances entre modelos sem cuidado

RF e GB calculam importancia de formas diferentes. Uma feature pode ser "top 1" em RF
e "top 10" em GB. Solucao: usar permutation importance para comparacao justa.

### Erro 7: Treinar GB sem early stopping

Sem early stopping, GB pode adicionar arvores demais (overfitting).
Solucao: usar validation_fraction e n_iter_no_change em GradientBoostingClassifier.

## 10. Resumo e Conexoes

### Hierarquia de Conceitos

```
ENSEMBLES EM ML
|
+-- Base: Arvore de Decisao (CART)
|     |-- Gini / Entropia (criterio de split)
|     |-- max_depth, min_samples (controle de complexidade)
|     |-- Feature importance (reducao de impureza)
|
+-- Bagging (paralelo)
|     |-- Bootstrap sampling
|     |-- Random Forest (+ feature sampling)
|     |-- OOB Score (validacao gratuita)
|
+-- Boosting (sequencial)
|     |-- AdaBoost (repondera amostras erradas)
|     |-- Gradient Boosting (ajusta residuos)
|     |-- XGBoost/LightGBM (estado da arte)
|
+-- Stacking (meta-learning)
      |-- Modelos base diversos
      |-- Meta-modelo combina predicoes
      |-- Maximo de performance
```

### Tabela de Conexoes

| Conceito | Onde apareceu antes | Onde sera usado |
|----------|-------------------|-----------------|
| Gini / Entropia | 0.2 (Probabilidade) | Base de todos os splits |
| Bias-Variance | 0.8 (Validacao) | Justifica ensembles |
| Bootstrap | 0.5 (Amostragem) | Bagging, RF |
| Feature Importance | 3.1, 3.2 | Selecao de features |
| Cross-Validation | 3.1 (CV) | Grid Search, OOB |
| Gradient Descent | 0.4 (Otimizacao) | Gradient Boosting |

### Checklist de Competencias

- [ ] Sei explicar o algoritmo CART (Gini, splits recursivos)
- [ ] Entendo como e por que arvores overfitam
- [ ] Sei a diferenca entre Bagging e Boosting
- [ ] Consigo treinar e avaliar Random Forest
- [ ] Entendo AdaBoost (repondera amostras) vs Gradient Boosting (ajusta residuos)
- [ ] Sei quando usar stacking e como montar modelos base diversos
- [ ] Consigo interpretar OOB score e feature importances

### Proximos Passos

1. **3.4 SVM e Kernels:** Outro paradigma de classificacao (margem maxima)
2. **3.5 Clustering:** Ensembles em contexto nao-supervisionado
3. **XGBoost/LightGBM:** Variantes otimizadas de Gradient Boosting (industria)
4. **Feature Selection:** Usar importances para selecionar features relevantes